# Population Data Processing for RAG

Loads **all 43,785 rows** (real + synthetic, all 292 users) from `daily_features` joined with `daily_logs` (phase, raw hormones) and `wearable_daily` (wearable signals).

Computes group-level population summaries, embeds with OpenAI `text-embedding-3-small`, and stores in ChromaDB collection `population_summaries`.

**Anomaly handling:**
- `resting_hr == 0` → treated as null (device not recording)
- `sleep_duration_min > 720` → treated as null (source data quality issue)

**Summary documents generated (~32 total):**
- 4 × phase-level summaries (all features)
- 7 × cycle-day bin profiles
- 8 × symptom cross-phase comparisons
- 9 × wearable signal norms
- 1 × cycle regularity
- 1 × user demographics
- 2 × data source comparison (real vs synthetic)


## Step 1 — Installation
Install required packages if running for the first time.
- `chromadb`: local vector database for storing and querying embeddings
- `openai`: for generating text embeddings via `text-embedding-3-small`
- `python-dotenv`: secure loading of API keys from `.env` file

In [49]:
# Run once
# !pip install chromadb openai pandas numpy python-dotenv


## Step 2 — Imports & Configuration
Load API key from `.env` (fallback to interactive prompt), connect to `herwell.db`, and define constants:
- `DB_PATH`: SQLite database with 43,785 daily records (42 real + 250 synthetic users)
- `CHROMA_DIR`: local ChromaDB persistent storage directory
- `EMBED_MODEL`: `text-embedding-3-small` (1536-dim vectors)

In [50]:
import sqlite3
import pandas as pd
import numpy as np
import os, json, time
from pathlib import Path
import chromadb
from openai import OpenAI
from dotenv import load_dotenv

# ── Load API key (3-layer fallback) ──────────────────────────────────────
# Layer 1: .env file in project directory (recommended for local dev)
load_dotenv(dotenv_path='D:/任梓嘉/NUS/sem2/Innovation Challenge/User data for RAG system/.env')

# Layer 2: system environment variable
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')

# Layer 3: interactive prompt (if neither above is set)
if not OPENAI_API_KEY:
    import getpass
    OPENAI_API_KEY = getpass.getpass('Enter OpenAI API Key: ')

assert OPENAI_API_KEY, "API key not found"
print(f'API key loaded: sk-...{OPENAI_API_KEY[-4:]}')

DB_PATH = 'D:/任梓嘉/NUS/sem2/Innovation Challenge/User Database/herwell.db'
CHROMA_DIR      = 'D:/任梓嘉/NUS/sem2/Innovation Challenge/User data for RAG system/chroma_db'
COLLECTION_NAME = 'population_summaries'
EMBED_MODEL     = 'text-embedding-3-small'

client_oai = OpenAI(api_key=OPENAI_API_KEY)
PHASE_MAP  = {0: 'Menstrual', 1: 'Follicular', 2: 'Fertility', 3: 'Luteal'}
PHASES     = ['Menstrual', 'Follicular', 'Fertility', 'Luteal']
print('Imports OK')


API key loaded: sk-...zn4A
Imports OK


## Step 3 — Load Data from Database
Execute a 3-table JOIN across `daily_features`, `daily_logs`, and `wearable_daily`.

**Data cleaning applied:**
- `resting_hr == 0` → `NaN` (device not recording)
- `sleep_duration_min > 720` → `NaN` (source data overflow)

Result: DataFrame `df` with all 43,785 rows and ~40 features per row.

In [51]:
conn = sqlite3.connect(DB_PATH)

# All users (real + synthetic)
df = pd.read_sql_query('''
    SELECT
        df.user_id, df.log_date,
        -- Cycle features
        df.cycle_start_flag, df.cycle_id,
        df.cycle_length_estimate, df.cycle_length_variation,
        df.days_since_last_period,
        -- Bleeding
        df.bleeding_present, df.bleeding_duration_days,
        df.flow_volume_num, df.heavy_flow_flag,
        -- Symptoms (Pipeline SEVERITY_MAP 0-5)
        df.pain_score, df.pain_trend,
        df.headache_score, df.fatigue_score, df.sleep_issue_score,
        df.mood_instability, df.stress_score, df.bloating_score,
        df.symptom_burden_score,
        -- Hormones
        df.lh_phase_z, df.estrogen_phase_z, df.pdg_phase_z,
        df.lh_anomaly_flag, df.estrogen_anomaly_flag,
        df.pdg_anomaly_flag, df.hormone_anomaly_flag,
        -- User-derived
        df.age, df.sexual_activity_flag,
        df.menstrual_health_literacy_num, df.early_menarche_flag,
        -- Cycle anomaly flags
        df.cycle_gt_35_flag, df.cycle_lt_21_flag, df.cycle_irregular_flag,
        -- From daily_logs
        dl.phase AS phase_int,
        dl.data_source,
        dl.lh AS lh_raw, dl.estrogen AS estrogen_raw, dl.pdg AS pdg_raw,
        dl.flow_color,
        dl.appetite, dl.exercise_level, dl.sore_breasts,
        dl.food_cravings, dl.indigestion,
        -- From wearable_daily
        w.resting_hr, w.nightly_temp, w.temp_diff_baseline,
        w.hrv_rmssd, w.sleep_score, w.sleep_duration_min,
        w.deep_sleep_min, w.rem_sleep_min, w.sleep_efficiency
    FROM daily_features df
    JOIN daily_logs dl
        ON df.user_id = dl.user_id AND df.log_date = dl.log_date
    LEFT JOIN wearable_daily w
        ON df.user_id = w.user_id AND df.log_date = w.log_date
    ORDER BY df.user_id, df.log_date
''', conn)

users = pd.read_sql_query('SELECT * FROM users', conn)
conn.close()

# Phase label
df['phase'] = df['phase_int'].map(PHASE_MAP)

# Clean known anomalies
df['resting_hr']       = df['resting_hr'].where(df['resting_hr'] > 0, np.nan)
df['sleep_duration_min'] = df['sleep_duration_min'].where(df['sleep_duration_min'] <= 720, np.nan)

real = df[df['data_source'] == 'dataset']
syn  = df[df['data_source'] == 'synthetic']

print(f'Total rows: {len(df):,}  |  real: {len(real):,}  |  synthetic: {len(syn):,}')
print(f'Users: {df["user_id"].nunique()}  |  real: {real["user_id"].nunique()}  |  synthetic: {syn["user_id"].nunique()}')
print(f'Phase distribution (all):\n{df["phase"].value_counts().to_string()}')


Total rows: 43,785  |  real: 5,659  |  synthetic: 38,126
Users: 292  |  real: 42  |  synthetic: 250
Phase distribution (all):
phase
Luteal        16821
Follicular    13305
Menstrual      7581
Fertility      6077


## Step 4 — Helper Functions
Utility functions used throughout the summary generation:
- `fmt()`: format floats with N/A fallback
- `pct()`: format percentages
- `pct_range()`: format p10/p50/p90 percentiles
- `severity_dist()`: compute % distribution across severity levels (none/low/moderate/high/severe)

In [52]:
def fmt(v, d=2):
    if v is None or (isinstance(v, float) and np.isnan(v)): return 'N/A'
    return f'{v:.{d}f}'

def pct(v):
    if v is None or (isinstance(v, float) and np.isnan(v)): return 'N/A'
    return f'{v*100:.1f}%'

def severity_dist(series):
    labels = {0:'none', 2:'low', 3:'moderate', 4:'high', 5:'severe'}
    counts = series.dropna().value_counts(normalize=True).sort_index()
    return ', '.join([f"{labels.get(int(k), str(int(k)))} {pct(v)}" for k, v in counts.items()])

def pct_range(series):
    s = series.dropna()
    if len(s) == 0: return 'N/A'
    return f'p10={fmt(s.quantile(0.1))}, p50={fmt(s.median())}, p90={fmt(s.quantile(0.9))}'

print('Helpers defined')


Helpers defined


## Step 5 — Phase × Feature Category Summaries (16 docs)
For each of the **4 menstrual phases** (Menstrual, Follicular, Fertility, Luteal), generate 4 focused documents:

| Doc | Content |
|-----|---------|
| `phase_{phase}_symptoms` | All 7 symptom scores with severity distribution |
| `phase_{phase}_hormones` | LH / Estrogen / PDG z-scores, anomaly rates, bleeding stats |
| `phase_{phase}_wearable` | HR, HRV, temperature, sleep (mean / std / p25 / p75) |
| `phase_{phase}_cycle` | Cycle length, irregularity, menarche, literacy flags |

**Total: 16 documents**

In [53]:
SYMPTOM_COLS  = ['pain_score','headache_score','fatigue_score','sleep_issue_score',
                 'mood_instability','stress_score','bloating_score']
HORMONE_COLS  = ['lh_phase_z','estrogen_phase_z','pdg_phase_z']
WEARABLE_COLS = ['resting_hr','hrv_rmssd','temp_diff_baseline','nightly_temp',
                 'sleep_score','sleep_duration_min','deep_sleep_min','rem_sleep_min','sleep_efficiency']
EXTRA_SYMPTOM = ['appetite','exercise_level','sore_breasts','food_cravings','indigestion']

phase_docs = []

for phase in PHASES:
    p   = df[df['phase'] == phase]
    p_r = p[p['data_source'] == 'dataset']
    n, n_r, n_u = len(p), len(p_r), p['user_id'].nunique()

    # Doc 1: Symptoms
    sym_lines = []
    for col in SYMPTOM_COLS:
        m, s = p[col].mean(), p[col].std()
        dist = severity_dist(p[col])
        sym_lines.append(f'  {col}: mean={fmt(m)}/5, std={fmt(s)} | distribution: {dist}')
    extra_lines = []
    for col in EXTRA_SYMPTOM:
        if col in p.columns:
            m = p[col].mean()
            extra_lines.append(f'  {col}: mean={fmt(m)}/4')
    phase_docs.append({
        'id': f'pop_phase_{phase.lower()}_symptoms',
        'text': (
            f'POPULATION SUMMARY --- {phase} Phase: Symptom Profile\n'
            f'Based on {n:,} daily records ({n_r:,} real + {n-n_r:,} synthetic), {n_u} users.\n\n'
            f'SYMPTOM SCORES (0=none, 2=low, 3=moderate, 4=high, 5=severe):\n'
            + '\n'.join(sym_lines)
            + f'\n  symptom_burden_score (mean of 6): mean={fmt(p["symptom_burden_score"].mean())}/5, '
            f'std={fmt(p["symptom_burden_score"].std())}\n'
            f'  pain_trend (5-day rolling avg): mean={fmt(p["pain_trend"].mean())}/5\n\n'
            f'ADDITIONAL SYMPTOMS (0-4 scale):\n'
            + '\n'.join(extra_lines) + '\n'
        ),
        'metadata': {'doc_type':'phase_symptoms','phase':phase,'n_records':n,'n_users':n_u}
    })

    # Doc 2: Hormones + Bleeding
    hor_lines = []
    for col in HORMONE_COLS:
        m, s = p[col].mean(), p[col].std()
        flag = col.replace('_phase_z','_anomaly_flag')
        anom = p[flag].mean() if flag in p.columns else float('nan')
        hor_lines.append(f'  {col}: mean={fmt(m)}, std={fmt(s)}, anomaly_rate={pct(anom)}')
    phase_docs.append({
        'id': f'pop_phase_{phase.lower()}_hormones',
        'text': (
            f'POPULATION SUMMARY --- {phase} Phase: Hormones & Bleeding\n'
            f'Based on {n:,} records ({n_r:,} real + {n-n_r:,} synthetic), {n_u} users.\n\n'
            f'HORMONES (within-phase z-scores, 0=phase average):\n'
            + '\n'.join(hor_lines)
            + f'\n  hormone_anomaly_flag rate: {pct(p["hormone_anomaly_flag"].mean())}\n\n'
            f'BLEEDING:\n'
            f'  bleeding_present: {pct(p["bleeding_present"].mean())} of days\n'
            f'  heavy_flow_flag: {pct(p["heavy_flow_flag"].mean())} of days\n'
            f'  avg bleeding_duration (when present): '
            f'{fmt(p.loc[p["bleeding_present"]==1,"bleeding_duration_days"].mean())} days\n'
            f'  flow_volume_num: mean={fmt(p["flow_volume_num"].mean())}/6 (0=none, 3=moderate, 6=heavy)\n'
        ),
        'metadata': {'doc_type':'phase_hormones','phase':phase,'n_records':n}
    })

    # Doc 3: Wearable signals
    wear_lines = []
    for col in WEARABLE_COLS:
        s_data = p[col].dropna()
        if len(s_data) > 0:
            wear_lines.append(
                f'  {col}: mean={fmt(s_data.mean())}, std={fmt(s_data.std())}, '
                f'p25={fmt(s_data.quantile(0.25))}, p75={fmt(s_data.quantile(0.75))}, '
                f'null_rate={pct(p[col].isna().mean())}'
            )
    phase_docs.append({
        'id': f'pop_phase_{phase.lower()}_wearable',
        'text': (
            f'POPULATION SUMMARY --- {phase} Phase: Wearable Signals\n'
            f'Based on {n:,} records ({n_r:,} real + {n-n_r:,} synthetic), {n_u} users.\n\n'
            f'WEARABLE MEASUREMENTS:\n'
            + '\n'.join(wear_lines) + '\n'
        ),
        'metadata': {'doc_type':'phase_wearable','phase':phase,'n_records':n}
    })

    # Doc 4: Cycle metrics
    phase_docs.append({
        'id': f'pop_phase_{phase.lower()}_cycle',
        'text': (
            f'POPULATION SUMMARY --- {phase} Phase: Cycle Metrics\n'
            f'Based on {n:,} records ({n_r:,} real + {n-n_r:,} synthetic), {n_u} users.\n\n'
            f'CYCLE METRICS:\n'
            f'  cycle_length_estimate: mean={fmt(p["cycle_length_estimate"].mean())} days, '
            f'std={fmt(p["cycle_length_estimate"].std())}, '
            f'{pct_range(p["cycle_length_estimate"])}\n'
            f'  days_since_last_period: mean={fmt(p["days_since_last_period"].mean())}, '
            f'{pct_range(p["days_since_last_period"])}\n'
            f'  cycle_gt_35_flag (long cycle): {pct(p["cycle_gt_35_flag"].mean())}\n'
            f'  cycle_lt_21_flag (short cycle): {pct(p["cycle_lt_21_flag"].mean())}\n'
            f'  cycle_irregular_flag (variation >7d): {pct(p["cycle_irregular_flag"].mean())}\n'
            f'  cycle_length_variation: mean={fmt(p["cycle_length_variation"].mean())} days\n'
            f'  early_menarche_flag (<11y): {pct(p["early_menarche_flag"].mean())}\n'
            f'  menstrual_health_literacy_num (1=Low,2=Mod,3=High): '
            f'mean={fmt(p["menstrual_health_literacy_num"].mean())}\n'
            f'  sexual_activity_flag: {pct(p["sexual_activity_flag"].mean())} active\n'
        ),
        'metadata': {'doc_type':'phase_cycle','phase':phase,'n_records':n}
    })

print(f'Generated {len(phase_docs)} phase docs (4 phases x 4 categories)')


Generated 16 phase docs (4 phases x 4 categories)


## Step 6 — Cycle Day Profiles (35 docs)
One document per cycle day (day 1-35), each describing the population-level distribution of:
symptoms, bleeding, hormones, and wearable signals on that specific cycle day.

Only days with >= 30 records are included. Each doc includes the typical phase distribution for that day.

**Total: up to 35 documents**

In [54]:
KEY_COLS = [
    'pain_score','fatigue_score','mood_instability','bloating_score','stress_score',
    'headache_score','sleep_issue_score','symptom_burden_score',
    'bleeding_present','heavy_flow_flag','flow_volume_num',
    'lh_phase_z','estrogen_phase_z','pdg_phase_z','hormone_anomaly_flag',
    'resting_hr','hrv_rmssd','temp_diff_baseline','sleep_score','sleep_duration_min',
    'sleep_efficiency','deep_sleep_min',
]

day_docs = []
for day in range(1, 36):
    mask = df['days_since_last_period'] == day
    p = df[mask]
    if len(p) < 30:
        continue
    n_r = (p['data_source'] == 'dataset').sum()
    phase_str = ', '.join([f'{ph} {pct(v)}'
                           for ph, v in p['phase'].value_counts(normalize=True).head(3).items()])
    lines = []
    for col in KEY_COLS:
        if col not in p.columns:
            continue
        s = p[col].dropna()
        if len(s) == 0:
            continue
        lines.append(f'  {col}: mean={fmt(s.mean())}, std={fmt(s.std())}, '
                     f'p25={fmt(s.quantile(0.25))}, p75={fmt(s.quantile(0.75))}')
    text = (
        f'POPULATION SUMMARY --- Cycle Day {day}\n'
        f'Based on {len(p):,} records ({n_r:,} real + {len(p)-n_r:,} synthetic), '
        f'{p["user_id"].nunique()} users.\n'
        f'Typical phase at day {day}: {phase_str}\n\n'
        f'KEY METRICS:\n' + '\n'.join(lines) + '\n'
    )
    day_docs.append({
        'id': f'pop_cycle_day_{day:02d}',
        'text': text,
        'metadata': {'doc_type':'cycle_day_profile','cycle_day':day,'n_records':len(p)}
    })

print(f'Generated {len(day_docs)} cycle-day docs (days 1-35, each with >=30 records)')


Generated 35 cycle-day docs (days 1-35, each with >=30 records)


## Step 7 — Per-Symptom Profiles (8 docs)
One detailed document per symptom, including:
- Overall mean and real-user mean
- Phase-by-phase breakdown with severity distribution
- Which phase has the highest burden

Symptoms: `pain_score`, `headache_score`, `fatigue_score`, `sleep_issue_score`, `mood_instability`, `stress_score`, `bloating_score`, `symptom_burden_score`

**Total: 8 documents**

In [55]:
symptom_meta = {
    'pain_score':         'menstrual cramps / pelvic pain',
    'headache_score':     'headaches',
    'fatigue_score':      'fatigue / low energy',
    'sleep_issue_score':  'sleep disturbances',
    'mood_instability':   'mood swings / emotional instability',
    'stress_score':       'stress',
    'bloating_score':     'bloating / abdominal discomfort',
    'symptom_burden_score': 'overall symptom burden (mean of 6 symptoms)',
}

sym_docs = []
for col, desc in symptom_meta.items():
    lines = []
    peak_phase, peak_val = None, -1
    for phase in PHASES:
        p = df[df['phase'] == phase]
        m, s = p[col].mean(), p[col].std()
        dist = severity_dist(p[col])
        lines.append(f'  {phase}: mean={fmt(m)}/5, std={fmt(s)} | {dist}')
        if not np.isnan(m) and m > peak_val:
            peak_val, peak_phase = m, phase

    text = (
        f'POPULATION SUMMARY — Symptom Profile: {desc} ({col})\n'
        f'Scale: 0=none, 2=low, 3=moderate, 4=high, 5=severe (Pipeline SEVERITY_MAP)\n'
        f'All users: mean={fmt(df[col].mean())}/5, std={fmt(df[col].std())}\n'
        f'Real users only: mean={fmt(df[df["data_source"]=="dataset"][col].mean())}/5\n'
        f'Highest in: {peak_phase} phase (mean={fmt(peak_val)}/5)\n\n'
        f'BY PHASE (all users):\n' + '\n'.join(lines) + '\n'
    )
    sym_docs.append({
        'id': f'pop_symptom_{col}',
        'text': text,
        'metadata': {'doc_type':'symptom_profile','symptom':col,'peak_phase':peak_phase}
    })

print(f'Generated {len(sym_docs)} symptom profile docs')


Generated 8 symptom profile docs


## Step 8 — Wearable Signal Norms (9 docs)
One document per wearable signal with phase-stratified statistics (mean, std, p10, p90, null rate):

`resting_hr`, `hrv_rmssd`, `temp_diff_baseline`, `nightly_temp`, `sleep_score`, `sleep_duration_min`, `deep_sleep_min`, `rem_sleep_min`, `sleep_efficiency`

**Total: 9 documents**

In [56]:
wear_meta = {
    'resting_hr':          ('bpm',   'Resting heart rate (zeros removed as invalid)'),
    'hrv_rmssd':           ('ms',    'Heart rate variability RMSSD (overnight)'),
    'temp_diff_baseline':  ('°C',    'Wrist temperature deviation from personal baseline'),
    'nightly_temp':        ('°C',    'Absolute wrist skin temperature at night'),
    'sleep_score':         ('0–100', 'Composite sleep quality score'),
    'sleep_duration_min':  ('min',   'Total nightly sleep duration (values >720 removed as invalid)'),
    'deep_sleep_min':      ('min',   'Deep sleep duration'),
    'rem_sleep_min':       ('min',   'REM sleep duration'),
    'sleep_efficiency':    ('%',     'Sleep efficiency (time asleep / time in bed)'),
}

wear_docs = []
for col, (unit, desc) in wear_meta.items():
    if col not in df.columns: continue
    lines = []
    for phase in PHASES:
        p = df[df['phase'] == phase]
        s = p[col].dropna()
        lines.append(
            f'  {phase}: mean={fmt(s.mean())} {unit}, std={fmt(s.std())}, '
            f'p10={fmt(s.quantile(0.1))}, p90={fmt(s.quantile(0.9))}, null_rate={pct(p[col].isna().mean())}'
        )
    overall = df[col].dropna()
    text = (
        f'POPULATION SUMMARY — Wearable Signal: {desc} ({col})\n'
        f'Unit: {unit} | Overall: mean={fmt(overall.mean())}, std={fmt(overall.std())}, '
        f'null_rate={pct(df[col].isna().mean())}\n\n'
        f'BY PHASE:\n' + '\n'.join(lines) + '\n'
    )
    wear_docs.append({
        'id': f'pop_wearable_{col}',
        'text': text,
        'metadata': {'doc_type':'wearable_norm','signal':col,'unit':unit}
    })

print(f'Generated {len(wear_docs)} wearable norm docs')


Generated 9 wearable norm docs


## Step 9 — Cycle Regularity, Demographics & Subgroup Analysis (10 docs)
**Cycle regularity** (1 doc): overall cycle length distribution, irregularity rates, variation stats.

**Demographics** (1 doc): age, menarche age, sexual activity, health literacy, ethnicity breakdown.

**Subgroup analysis** (8 docs): population profiles for:
- Regular vs Irregular cycle users
- High vs Low symptom burden (top/bottom 25%)
- Long cycle (>35d) and Short cycle (<21d) users
- Early menarche (<11y) users
- High vs Low menstrual health literacy

**Total: 10 documents**

In [57]:
# Cycle regularity
cycle_text = (
    f'POPULATION SUMMARY --- Menstrual Cycle Regularity\n'
    f'Based on {df["user_id"].nunique()} users ({df[df["data_source"]=="dataset"]["user_id"].nunique()} real, '
    f'{df[df["data_source"]=="synthetic"]["user_id"].nunique()} synthetic), {len(df):,} daily records.\n\n'
    f'CYCLE LENGTH (cycle_length_estimate):\n'
    f'  All users: mean={fmt(df["cycle_length_estimate"].mean())} days, '
    f'std={fmt(df["cycle_length_estimate"].std())}, '
    f'{pct_range(df["cycle_length_estimate"])}\n'
    f'  Real users: mean={fmt(df[df["data_source"]=="dataset"]["cycle_length_estimate"].mean())} days\n'
    f'  Long cycle (>35d): {pct(df["cycle_gt_35_flag"].mean())} of days\n'
    f'  Short cycle (<21d): {pct(df["cycle_lt_21_flag"].mean())} of days\n'
    f'  Irregular (variation >7d): {pct(df["cycle_irregular_flag"].mean())} of days\n\n'
    f'CYCLE LENGTH VARIATION (rolling 3-cycle std):\n'
    f'  mean={fmt(df["cycle_length_variation"].mean())} days, std={fmt(df["cycle_length_variation"].std())}\n\n'
    f'CYCLE ID: avg {fmt(df.groupby("user_id")["cycle_id"].max().mean())} cycles per user\n'
)
cycle_doc = {'id':'pop_cycle_regularity','text':cycle_text,
             'metadata':{'doc_type':'cycle_regularity'}}

# Demographics
real_users = users[users['id'] <= 1000]
demo_text = (
    f'POPULATION SUMMARY --- User Demographics\n'
    f'{len(users)} users total: {(users["id"]<=1000).sum()} real participants, '
    f'{(users["id"]>1000).sum()} synthetic.\n\n'
    f'REAL USERS (n={len(real_users)}):\n'
    f'  birth_year: {real_users["birth_year"].min()}-{real_users["birth_year"].max()}, '
    f'mean={real_users["birth_year"].mean():.0f}\n'
    f'  age (from daily_features): mean={fmt(df[df["data_source"]=="dataset"]["age"].mean())} yrs\n'
    f'  age_of_first_menarche: mean={fmt(real_users["age_of_first_menarche"].mean())}, '
    f'min={real_users["age_of_first_menarche"].min()}, max={real_users["age_of_first_menarche"].max()}\n'
    f'  early_menarche_flag (<11y): {pct(df[df["data_source"]=="dataset"]["early_menarche_flag"].mean())}\n'
    f'  sexually_active: '
    f'{pct((real_users["sexually_active"]==1).mean())} yes, '
    f'{pct((real_users["sexually_active"]==0).mean())} no\n'
    f'  menstrual_health_literacy: '
    f'Low {pct((real_users["menstrual_health_literacy"]==0).mean())}, '
    f'Medium {pct((real_users["menstrual_health_literacy"]==1).mean())}, '
    f'High {pct((real_users["menstrual_health_literacy"]==2).mean())}\n'
    f'  gender: {dict(real_users["gender"].value_counts().to_dict())}\n'
    f'  ethnicity top-3: {list(real_users["ethnicity"].value_counts().head(3).index)}\n\n'
    f'ALL USERS (real + synthetic):\n'
    f'  age mean={fmt(df["age"].mean())} yrs\n'
    f'  sexual_activity_flag: {pct(df["sexual_activity_flag"].mean())} active\n'
    f'  early_menarche_flag: {pct(df["early_menarche_flag"].mean())}\n'
    f'  menstrual_health_literacy_num (1=Low,2=Mod,3=High): '
    f'mean={fmt(df["menstrual_health_literacy_num"].mean())}\n'
)
demo_doc = {'id':'pop_demographics','text':demo_text,
            'metadata':{'doc_type':'demographics'}}

# ── Subgroup analysis ────────────────────────────────────────────────────
subgroup_docs = []

def subgroup_summary(sg, label, definition):
    n, n_r = len(sg), (sg['data_source']=='dataset').sum()
    n_u = sg['user_id'].nunique()
    phase_dist = ', '.join([f'{ph} {pct(v)}'
                            for ph, v in sg['phase'].value_counts(normalize=True).head(4).items()])
    sym_lines = '\n'.join([f'  {col}: mean={fmt(sg[col].mean())}/5' for col in SYMPTOM_COLS])
    return (
        f'SUBGROUP SUMMARY --- {label}\n'
        f'Definition: {definition}\n'
        f'Based on {n:,} records ({n_r:,} real + {n-n_r:,} synthetic), {n_u} users.\n\n'
        f'PHASE DISTRIBUTION: {phase_dist}\n\n'
        f'SYMPTOM SCORES (0-5):\n{sym_lines}\n'
        f'  symptom_burden_score: mean={fmt(sg["symptom_burden_score"].mean())}/5\n\n'
        f'CYCLE METRICS:\n'
        f'  cycle_length_estimate: mean={fmt(sg["cycle_length_estimate"].mean())} days, '
        f'std={fmt(sg["cycle_length_estimate"].std())}\n'
        f'  cycle_irregular_flag: {pct(sg["cycle_irregular_flag"].mean())}\n'
        f'  early_menarche_flag: {pct(sg["early_menarche_flag"].mean())}\n\n'
        f'WEARABLE SIGNALS:\n'
        f'  resting_hr: mean={fmt(sg["resting_hr"].mean())} bpm\n'
        f'  hrv_rmssd: mean={fmt(sg["hrv_rmssd"].mean())} ms\n'
        f'  sleep_score: mean={fmt(sg["sleep_score"].mean())}\n'
        f'  sleep_duration_min: mean={fmt(sg["sleep_duration_min"].mean())} min\n'
    )

# 1. Regular vs Irregular cycle
for flag_val, label, defn, key in [
    (0, 'Regular Menstrual Cycle Users',   'cycle_irregular_flag=0, variation <=7 days', 'regular_cycle'),
    (1, 'Irregular Menstrual Cycle Users',  'cycle_irregular_flag=1, variation >7 days',  'irregular_cycle'),
]:
    sg = df[df['cycle_irregular_flag'] == flag_val]
    subgroup_docs.append({
        'id': f'pop_subgroup_{key}',
        'text': subgroup_summary(sg, label, defn),
        'metadata': {'doc_type':'subgroup','subgroup':key}
    })

# 2. High vs Low symptom burden (quartiles)
q75 = df['symptom_burden_score'].quantile(0.75)
q25 = df['symptom_burden_score'].quantile(0.25)
for sg, label, defn, key in [
    (df[df['symptom_burden_score'] >= q75], 'High Symptom Burden Users',
     'symptom_burden_score >= top 25%', 'high_burden'),
    (df[df['symptom_burden_score'] <= q25], 'Low Symptom Burden Users',
     'symptom_burden_score <= bottom 25%', 'low_burden'),
]:
    subgroup_docs.append({
        'id': f'pop_subgroup_{key}',
        'text': subgroup_summary(sg, label, defn),
        'metadata': {'doc_type':'subgroup','subgroup':key}
    })

# 3. Long / Short cycle
for flag, label, defn, key in [
    ('cycle_gt_35_flag', 'Long Cycle Users (>35 days)',  'cycle_gt_35_flag=1', 'long_cycle'),
    ('cycle_lt_21_flag', 'Short Cycle Users (<21 days)', 'cycle_lt_21_flag=1', 'short_cycle'),
]:
    sg = df[df[flag] == 1]
    if len(sg) >= 50:
        subgroup_docs.append({
            'id': f'pop_subgroup_{key}',
            'text': subgroup_summary(sg, label, defn),
            'metadata': {'doc_type':'subgroup','subgroup':key}
        })

# 4. Early menarche
sg = df[df['early_menarche_flag'] == 1]
if len(sg) >= 50:
    subgroup_docs.append({
        'id': 'pop_subgroup_early_menarche',
        'text': subgroup_summary(sg, 'Early Menarche Users (<11 years old)', 'early_menarche_flag=1'),
        'metadata': {'doc_type':'subgroup','subgroup':'early_menarche'}
    })

# 5. High vs Low health literacy
for lit_val, label, key in [
    (3, 'High Menstrual Health Literacy Users', 'high_literacy'),
    (1, 'Low Menstrual Health Literacy Users',  'low_literacy'),
]:
    sg = df[df['menstrual_health_literacy_num'] == lit_val]
    if len(sg) >= 50:
        subgroup_docs.append({
            'id': f'pop_subgroup_{key}',
            'text': subgroup_summary(sg, label, f'menstrual_health_literacy_num={lit_val}'),
            'metadata': {'doc_type':'subgroup','subgroup':key}
        })

print(f'Generated cycle_doc + demo_doc + {len(subgroup_docs)} subgroup docs')
print('Cycle doc preview:')
print(cycle_text[:300])


Generated cycle_doc + demo_doc + 8 subgroup docs
Cycle doc preview:
POPULATION SUMMARY --- Menstrual Cycle Regularity
Based on 292 users (42 real, 250 synthetic), 43,785 daily records.

CYCLE LENGTH (cycle_length_estimate):
  All users: mean=31.65 days, std=3.07, p10=28.00, p50=32.00, p90=35.00
  Real users: mean=30.22 days
  Long cycle (>35d): 8.3% of days
  Short 


## Step 10 — Cross-cutting Patterns (4 docs)
**Real vs Synthetic comparison** (2 docs): symptom means and cycle/wearable metrics compared.

**Symptom co-occurrence** (1 doc): when symptom X >= 3, which other symptoms are also elevated?

**Wearable x Symptom correlation** (1 doc): symptom profiles by HRV quartile, sleep score quartile, and temperature deviation.

**Total: 4 documents**

In [58]:
# Real vs synthetic comparison
compare_docs = []
sym_compare_lines = []
for col in SYMPTOM_COLS + ['symptom_burden_score','flow_volume_num','heavy_flow_flag']:
    r_mean = df[df['data_source']=='dataset'][col].mean()
    s_mean = df[df['data_source']=='synthetic'][col].mean()
    sym_compare_lines.append(f'  {col}: real={fmt(r_mean)}, synthetic={fmt(s_mean)}')

compare_docs.append({
    'id': 'pop_real_vs_synthetic_symptoms',
    'text': (
        'POPULATION SUMMARY --- Real vs Synthetic: Symptom Comparison\n'
        f'Real: {len(real):,} rows, {real["user_id"].nunique()} users | '
        f'Synthetic: {len(syn):,} rows, {syn["user_id"].nunique()} users\n\n'
        'SYMPTOM MEANS (Pipeline SEVERITY_MAP 0-5):\n'
        + '\n'.join(sym_compare_lines) + '\n'
    ),
    'metadata': {'doc_type':'real_vs_synthetic','topic':'symptoms'}
})

cycle_compare_lines = []
for col in ['cycle_length_estimate','days_since_last_period','cycle_length_variation',
            'resting_hr','hrv_rmssd','sleep_score','sleep_duration_min']:
    r_m = df[df['data_source']=='dataset'][col].mean()
    s_m = df[df['data_source']=='synthetic'][col].mean()
    cycle_compare_lines.append(f'  {col}: real={fmt(r_m)}, synthetic={fmt(s_m)}')

compare_docs.append({
    'id': 'pop_real_vs_synthetic_cycle_wearable',
    'text': (
        'POPULATION SUMMARY --- Real vs Synthetic: Cycle & Wearable Comparison\n\n'
        'CYCLE AND WEARABLE MEANS:\n'
        + '\n'.join(cycle_compare_lines) + '\n\n'
        'Note: Synthetic data mirrors real null rates and phase-conditional distributions.\n'
    ),
    'metadata': {'doc_type':'real_vs_synthetic','topic':'cycle_wearable'}
})

# ── Symptom co-occurrence ──────────────────────────────────────────────────
cooccur_lines = []
for col in SYMPTOM_COLS:
    high = df[df[col] >= 3]
    if len(high) < 100:
        continue
    others = []
    for other in SYMPTOM_COLS:
        if other == col:
            continue
        baseline = df[other].mean()
        elevated = high[other].mean()
        if elevated > baseline * 1.2:
            others.append(f'{other} (mean={fmt(elevated)}/5 vs baseline {fmt(baseline)}/5)')
    cooccur_lines.append(
        f'  When {col}>=3: co-elevated -> ' + (', '.join(others) if others else 'none significant')
    )

burden_by_phase = '\n'.join([
    f'  {ph}: symptom_burden_score mean={fmt(df[df["phase"]==ph]["symptom_burden_score"].mean())}/5'
    for ph in PHASES
])
cooccur_doc = {
    'id': 'pop_symptom_cooccurrence',
    'text': (
        'POPULATION SUMMARY --- Symptom Co-occurrence Patterns\n'
        'When one symptom is elevated (score >=3), which other symptoms tend to be elevated?\n\n'
        + '\n'.join(cooccur_lines) + '\n\n'
        'PHASE-SPECIFIC SYMPTOM BURDEN:\n'
        + burden_by_phase + '\n'
    ),
    'metadata': {'doc_type':'symptom_cooccurrence'}
}

# ── Wearable x symptom correlations ────────────────────────────────────────
hrv_q25  = df['hrv_rmssd'].quantile(0.25)
hrv_q75  = df['hrv_rmssd'].quantile(0.75)
slp_q25  = df['sleep_score'].quantile(0.25)
slp_q75  = df['sleep_score'].quantile(0.75)
temp_q75 = df['temp_diff_baseline'].quantile(0.75)
temp_q25 = df['temp_diff_baseline'].quantile(0.25)

def sym_snap(g):
    return ', '.join([
        f'{c}={fmt(g[c].mean())}/5'
        for c in ['pain_score','fatigue_score','mood_instability','stress_score','bloating_score']
    ])

low_hrv    = df[df['hrv_rmssd'] <= hrv_q25]
high_hrv   = df[df['hrv_rmssd'] >= hrv_q75]
low_sleep  = df[df['sleep_score'] <= slp_q25]
high_sleep = df[df['sleep_score'] >= slp_q75]
high_temp  = df[df['temp_diff_baseline'] >= temp_q75]
low_temp   = df[df['temp_diff_baseline'] <= temp_q25]

low_hrv_phases   = ', '.join([f'{ph} {pct(v)}' for ph,v in low_hrv['phase'].value_counts(normalize=True).head(3).items()])
high_temp_phases = ', '.join([f'{ph} {pct(v)}' for ph,v in high_temp['phase'].value_counts(normalize=True).head(2).items()])
low_temp_phases  = ', '.join([f'{ph} {pct(v)}' for ph,v in low_temp['phase'].value_counts(normalize=True).head(2).items()])

wear_sym_doc = {
    'id': 'pop_wearable_symptom_correlation',
    'text': (
        f'POPULATION SUMMARY --- Wearable Signals and Symptom Correlations\n\n'
        f'LOW HRV (<=p25={fmt(hrv_q25)} ms, n={len(low_hrv):,}):\n'
        f'  Symptoms: {sym_snap(low_hrv)}\n'
        f'  symptom_burden_score: mean={fmt(low_hrv["symptom_burden_score"].mean())}/5\n'
        f'  Phase: {low_hrv_phases}\n\n'
        f'HIGH HRV (>=p75={fmt(hrv_q75)} ms, n={len(high_hrv):,}):\n'
        f'  Symptoms: {sym_snap(high_hrv)}\n'
        f'  symptom_burden_score: mean={fmt(high_hrv["symptom_burden_score"].mean())}/5\n\n'
        f'POOR SLEEP (score <=p25={fmt(slp_q25)}, n={len(low_sleep):,}):\n'
        f'  Symptoms: {sym_snap(low_sleep)}\n'
        f'  symptom_burden_score: mean={fmt(low_sleep["symptom_burden_score"].mean())}/5\n\n'
        f'GOOD SLEEP (score >=p75={fmt(slp_q75)}, n={len(high_sleep):,}):\n'
        f'  Symptoms: {sym_snap(high_sleep)}\n'
        f'  symptom_burden_score: mean={fmt(high_sleep["symptom_burden_score"].mean())}/5\n\n'
        f'ELEVATED BODY TEMP (>=p75={fmt(temp_q75)} C, n={len(high_temp):,}):\n'
        f'  Typical phase: {high_temp_phases}\n'
        f'  Symptoms: {sym_snap(high_temp)}\n'
        f'  bleeding_present: {pct(high_temp["bleeding_present"].mean())}\n\n'
        f'LOW BODY TEMP (<=p25={fmt(temp_q25)} C, n={len(low_temp):,}):\n'
        f'  Typical phase: {low_temp_phases}\n'
        f'  Symptoms: {sym_snap(low_temp)}\n'
        f'  bleeding_present: {pct(low_temp["bleeding_present"].mean())}\n'
    ),
    'metadata': {'doc_type':'wearable_symptom_correlation'}
}

print(f'Generated {len(compare_docs)} compare docs + cooccur doc + wearable-symptom doc')


Generated 2 compare docs + cooccur doc + wearable-symptom doc


## Step 11 — Combine All Documents
Merge all document lists into `all_docs`. Print a breakdown by document type and word count per document.

**Grand total: 82 documents**

In [59]:
all_docs = (phase_docs + day_docs + sym_docs + wear_docs +
           [cycle_doc, demo_doc] + subgroup_docs + compare_docs +
           [cooccur_doc, wear_sym_doc])
print(f'Total population summary documents: {len(all_docs)}')
counts = {}
for d in all_docs:
    t = d['metadata']['doc_type']
    counts[t] = counts.get(t, 0) + 1
for t, c in sorted(counts.items()):
    print(f'  {t:<35} {c:>3} docs')
print()
for d in all_docs:
    print(f"  {d['id']:<55} {len(d['text'].split()):>5} words")


Total population summary documents: 82
  cycle_day_profile                    35 docs
  cycle_regularity                      1 docs
  demographics                          1 docs
  phase_cycle                           4 docs
  phase_hormones                        4 docs
  phase_symptoms                        4 docs
  phase_wearable                        4 docs
  real_vs_synthetic                     2 docs
  subgroup                              8 docs
  symptom_cooccurrence                  1 docs
  symptom_profile                       8 docs
  wearable_norm                         9 docs
  wearable_symptom_correlation          1 docs

  pop_phase_menstrual_symptoms                              156 words
  pop_phase_menstrual_hormones                               59 words
  pop_phase_menstrual_wearable                               74 words
  pop_phase_menstrual_cycle                                  56 words
  pop_phase_follicular_symptoms                             156 words

## Step 12 — Generate Embeddings
Batch-embed all document texts using OpenAI `text-embedding-3-small` (1536 dimensions).
Documents are sent in batches of 50 to stay within API rate limits.
Total API calls: ceil(82 / 50) = 2 batches.

In [60]:
def embed_texts(texts, model=EMBED_MODEL, batch_size=50):
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        resp  = client_oai.embeddings.create(input=batch, model=model)
        vectors.extend([r.embedding for r in resp.data])
        print(f'  Embedded {min(i+batch_size, len(texts))}/{len(texts)}')
        time.sleep(0.3)
    return vectors

print(f'Embedding {len(all_docs)} documents with {EMBED_MODEL}...')
vectors = embed_texts([d['text'] for d in all_docs])
print(f'Done. Vector dim: {len(vectors[0])}')


Embedding 82 documents with text-embedding-3-small...
  Embedded 50/82
  Embedded 82/82
Done. Vector dim: 1536


## Step 13 — Store in ChromaDB
Create (or reset) the `population_summaries` collection in the local ChromaDB instance.
Each document is stored with its text, 1536-dim embedding vector, and metadata.

Storage: `chroma_db/chroma.sqlite3` (~1.6 MB)

In [61]:
Path(CHROMA_DIR).mkdir(parents=True, exist_ok=True)
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)

try:
    chroma_client.delete_collection(COLLECTION_NAME)
    print(f'Deleted existing: {COLLECTION_NAME}')
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={'hnsw:space': 'cosine'}
)
collection.add(
    ids        = [d['id'] for d in all_docs],
    documents  = [d['text'] for d in all_docs],
    embeddings = vectors,
    metadatas  = [d['metadata'] for d in all_docs],
)
print(f'Stored {collection.count()} documents in ChromaDB at {CHROMA_DIR}')


Deleted existing: population_summaries
Stored 82 documents in ChromaDB at D:/任梓嘉/NUS/sem2/Innovation Challenge/User data for RAG system/chroma_db


## Step 14 — Define Query Function & Smoke Test
`query_population(question, n_results=3)`: embeds the question and retrieves the most semantically similar documents.

Returns: list of `{id, text, metadata, distance}` (L2 distance — lower = more relevant).

4 smoke-test queries verify the pipeline end-to-end.

In [62]:
def query_population(question, n_results=3, where=None):
    q_vec = client_oai.embeddings.create(input=[question], model=EMBED_MODEL).data[0].embedding
    kwargs = dict(query_embeddings=[q_vec], n_results=n_results,
                  include=['documents','metadatas','distances'])
    if where: kwargs['where'] = where
    res = collection.query(**kwargs)
    results = []
    for doc, meta, dist in zip(res['documents'][0], res['metadatas'][0], res['distances'][0]):
        results.append({'id': meta.get('doc_type',''), 'text': doc, 'metadata': meta, 'distance': dist})
    return results

def print_results(results):
    for r in results:
        print(f"--- dist={r['distance']:.4f}, type={r['metadata']['doc_type']} ---")
        print(r['text'][:500])
        print()

print('=== Q1: pain during luteal phase ===')
print_results(query_population('What is the typical pain level during the luteal phase?'))

print('=== Q2: normal cycle length ===')
print_results(query_population('Is a 32-day menstrual cycle considered normal?'))

print('=== Q3: sleep quality across cycle ===')
print_results(query_population('How does sleep quality change across the menstrual cycle?'))

print('=== Q4: ovulation symptoms ===')
print_results(query_population('What symptoms are typical around ovulation days 14-16?'))


=== Q1: pain during luteal phase ===
--- dist=0.3794, type=phase_symptoms ---
POPULATION SUMMARY --- Luteal Phase: Symptom Profile
Based on 16,821 daily records (1,912 real + 14,909 synthetic), 292 users.

SYMPTOM SCORES (0=none, 2=low, 3=moderate, 4=high, 5=severe):
  pain_score: mean=0.68/5, std=1.28 | distribution: none 75.9%, low 9.3%, moderate 10.3%, high 3.5%, severe 1.0%
  headache_score: mean=1.06/5, std=1.49 | distribution: none 63.4%, low 14.5%, moderate 13.5%, high 6.7%, severe 1.9%
  fatigue_score: mean=2.33/5, std=1.62 | distribution: none 27.2%, low 17.7%, m

--- dist=0.4190, type=symptom_profile ---
POPULATION SUMMARY — Symptom Profile: menstrual cramps / pelvic pain (pain_score)
Scale: 0=none, 2=low, 3=moderate, 4=high, 5=severe (Pipeline SEVERITY_MAP)
All users: mean=0.70/5, std=1.33
Real users only: mean=0.72/5
Highest in: Menstrual phase (mean=1.59/5)

BY PHASE (all users):
  Menstrual: mean=1.59/5, std=1.76 | none 50.9%, low 12.8%, moderate 17.6%, high 12.5%, severe

## Step 15 — Export to JSON
Export all documents (text + metadata, without vectors) to `population_summaries.json`.
Useful for inspection or reloading without re-embedding.

In [63]:
OUT = 'D:/任梓嘉/NUS/sem2/Innovation Challenge/User data for RAG system/population_summaries.json'
with open(OUT, 'w', encoding='utf-8') as f:
    json.dump([{'id':d['id'],'metadata':d['metadata'],'text':d['text']}
               for d in all_docs], f, ensure_ascii=False, indent=2)
print(f'Exported {len(all_docs)} docs to {OUT}')


Exported 82 docs to D:/任梓嘉/NUS/sem2/Innovation Challenge/User data for RAG system/population_summaries.json


## Step 16 — Retrieval Quality Evaluation
8 targeted test queries covering phase symptoms, wearable signals, cycle norms, irregularity, and cross-signal correlations.

For each query: top-3 results with document ID, L2 distance, and content snippet.

**Distance benchmarks:** < 0.40 excellent | 0.40-0.55 good | > 0.55 off-topic

In [64]:
# ── Retrieval Quality Test ────────────────────────────────────────────────
test_queries = [
    'Why am I so exhausted during the luteal phase?',
    'My HRV is very low, is it related to my symptoms?',
    'How bad should cramps be on cycle day 3?',
    'My cycle is irregular, does it affect symptom severity?',
    'What are typical body changes during ovulation / fertility phase?',
    'When I sleep poorly, how does it affect my mood?',
    'What is a normal resting heart rate during menstruation?',
    'Is heavy flow common? What counts as heavy?',
]

print('=' * 70)
for q in test_queries:
    results = query_population(q, n_results=3)
    print(f'\nQ: {q}')
    for i, r in enumerate(results, 1):
        dist = r.get('distance', float('nan'))
        doc_type = r['metadata'].get('doc_type', '')
        phase    = r['metadata'].get('phase', r['metadata'].get('subgroup', r['metadata'].get('cycle_day', '')))
        snippet  = r['text'][:120].replace('\n', ' ').strip()
        print(f'  [{i}] id={r["id"]:<45} dist={dist:.3f}')
        print(f'       type={doc_type}, extra={phase}')
        print(f'       "{snippet}..."')
print('=' * 70)



Q: Why am I so exhausted during the luteal phase?
  [1] id=phase_symptoms                                dist=0.480
       type=phase_symptoms, extra=Luteal
       "POPULATION SUMMARY --- Luteal Phase: Symptom Profile Based on 16,821 daily records (1,912 real + 14,909 synthetic), 292..."
  [2] id=symptom_profile                               dist=0.486
       type=symptom_profile, extra=
       "POPULATION SUMMARY — Symptom Profile: fatigue / low energy (fatigue_score) Scale: 0=none, 2=low, 3=moderate, 4=high, 5=s..."
  [3] id=phase_hormones                                dist=0.508
       type=phase_hormones, extra=Luteal
       "POPULATION SUMMARY --- Luteal Phase: Hormones & Bleeding Based on 16,821 records (1,912 real + 14,909 synthetic), 292 us..."

Q: My HRV is very low, is it related to my symptoms?
  [1] id=wearable_symptom_correlation                  dist=0.372
       type=wearable_symptom_correlation, extra=
       "POPULATION SUMMARY --- Wearable Signals and Symptom Correl